In [ ]:
# Lab type: review
# Course: AI401 — AI Applications with LLMs
# Lesson: Context Window Architecture: What Goes Where and Why
# Task: Review a RAG prompt builder and answer judgment questions about token budget allocation

In [ ]:
# Install the Anthropic library
!pip install anthropic

To use the Anthropic API, you'll need an API key. If you don't already have one, create a key on the [Anthropic console](https://console.anthropic.com/settings/keys).

In Colab, add the key to the secrets manager under the "🔑" in the left panel. Give it the name `ANTHROPIC_API_KEY`. Then pass the key to the client initialization.

In [ ]:
import os

try:
    # Attempt to import google.colab.userdata, which is only available in Colab
    from google.colab import userdata

    # Fetch the API key from Colab's secrets manager
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    print("Anthropic API key loaded from Colab secrets.")
except ImportError:
    # If not in Colab, try to load from a .env file using python-dotenv
    try:
        from dotenv import load_dotenv
        load_dotenv()
        ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
        if ANTHROPIC_API_KEY:
            print("Anthropic API key loaded from .env file.")
        else:
            print("Anthropic API key not found in .env file. Please ensure ANTHROPIC_API_KEY is set.")
    except ImportError:
        print("python-dotenv not installed. Please install it (`pip install python-dotenv`) or ensure ANTHROPIC_API_KEY is set as an environment variable.")
        ANTHROPIC_API_KEY = None

# Ensure the API key is not None before proceeding, or handle the error appropriately
if ANTHROPIC_API_KEY is None:
    raise ValueError("ANTHROPIC_API_KEY is not set. Please set it in Colab secrets or a .env file.")

# Lab: Reviewing a RAG Prompt Builder

The implementation below assembles retrieved chunks into a prompt for a retrieval-augmented generation (RAG) pipeline. The code is correct and runs without errors.

Your task: read the code, run the diagnostic cells, and answer the judgment questions in the markdown cells. Write your answers in the blank response cells.

## Setup

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

SYSTEM_PROMPT = (
    "Answer the question based only on the provided context. "
    "If the context does not contain enough information to answer, "
    "say so explicitly. "
    "Do not combine contradictory information — identify the contradiction instead."
)

## The implementation

In [ ]:
def estimate_tokens(text: str) -> int:
    """Rough token estimate: 1 token ≈ 4 characters."""
    return len(text) // 4


def truncate_chunks(
    chunks: list[str],
    max_tokens: int,
) -> list[str]:
    """
    Keep as many chunks as fit in max_tokens, dropping from the end.
    chunks should be ordered by relevance score descending (most relevant first).
    """
    selected = []
    used = 0
    for chunk in chunks:
        t = estimate_tokens(chunk)
        if used + t > max_tokens:
            break
        selected.append(chunk)
        used += t
    return selected


def build_rag_prompt(
    question: str,
    retrieved_chunks: list[str],
    max_context_tokens: int = 3000,
) -> str:
    """
    Assemble retrieved chunks into a user-turn prompt.
    retrieved_chunks: ordered by relevance score descending.
    """
    selected = truncate_chunks(retrieved_chunks, max_context_tokens)
    context_block = "\n\n---\n\n".join(selected)
    return f"Relevant context:\n\n{context_block}\n\nQuestion: {question}"

## Inspection: token budget

In [ ]:
# Sample data — representative of what the pipeline processes in production
question = "What is the refund policy for digital subscriptions?"

retrieved_chunks = [
    # Chunk 0 — most relevant (score 0.92)
    "Digital subscription refunds are processed within 5 business days. "
    "Customers must request a refund within 14 days of purchase. "
    "Refunds are not available after the subscription content has been accessed more than once.",

    # Chunk 1 — relevant (score 0.81)
    "Our refund policy applies to all products sold on the platform. "
    "Physical goods must be returned within 30 days. "
    "Digital products have a separate policy — see the Digital Subscription Terms.",

    # Chunk 2 — borderline relevant (score 0.60)
    "Subscription tiers: Basic ($9/mo), Professional ($29/mo), Enterprise (custom). "
    "All tiers include a 7-day free trial. Annual billing available at 20% discount.",

    # Chunk 3 — marginally relevant (score 0.41)
    "Customer support hours: Monday to Friday, 9am–6pm GMT. "
    "For urgent issues, use the priority support channel available to Professional and Enterprise subscribers.",
]

# Build the prompt
prompt = build_rag_prompt(question, retrieved_chunks, max_context_tokens=3000)

# Count tokens with the real API before making a call
token_count = client.messages.count_tokens(
    model="claude-haiku-4-5-20251001",
    system=SYSTEM_PROMPT,
    messages=[{"role": "user", "content": prompt}],
)

print(f"System prompt tokens : {estimate_tokens(SYSTEM_PROMPT)}")
print(f"User prompt tokens   : {estimate_tokens(prompt)}")
print(f"API token count      : {token_count.input_tokens}")
print(f"Chunks selected      : {len(truncate_chunks(retrieved_chunks, 3000))} / {len(retrieved_chunks)}")

## Judgment question 1

> `truncate_chunks()` drops chunks from the **end** of the list. In this implementation, chunks are already ordered by relevance descending, so the least relevant chunk is dropped first. When would **end-truncation** be the wrong strategy — and what should replace it?

In [ ]:
# Your answer (write in this cell as a comment):
#
#

<details>
<summary>🔑 Reveal answer — Q1</summary>

**When end-truncation is wrong:** When chunks are not pre-sorted by relevance score descending. If the retrieved list is ordered by recency, insertion order, or any other criterion, end-truncation silently drops the most relevant chunk rather than the least relevant one — the RAG pipeline answers from lower-quality context without any error signal.

**What should replace it:** Sort chunks by relevance score descending before truncating, then apply end-truncation. This guarantees that the dropped chunk is always the weakest match. For cases where ordering is uncertain, a budget-aware selection algorithm (e.g., greedy selection by score until the token budget is exhausted) is more robust than position-based dropping.

</details>

## Inspection: contradiction handling

In [ ]:
# Introduce a contradictory chunk — simulates stale and updated policy both in the vector store
contradictory_chunks = [
    "Digital subscription refunds are processed within 5 business days. "
    "Customers must request a refund within 14 days of purchase.",

    "As of Q1 2025, the refund window for digital subscriptions was extended to 30 days. "
    "The 14-day policy is no longer in effect.",
]

prompt_with_contradiction = build_rag_prompt(question, contradictory_chunks)
print(prompt_with_contradiction)

## Judgment question 2

> The system prompt instructs the model: *'Do not combine contradictory information — identify the contradiction instead.'* The two chunks above give different refund windows (14 days vs. 30 days).

> (a) What answer should a correctly-behaving model give for this input?  
> (b) What answer might a model give if the contradiction instruction were absent?  
> (c) How would you detect in production whether the model is handling contradictions correctly?

In [ ]:
# Your answer:
#
# (a)
#
# (b)
#
# (c)
#

<details>
<summary>🔑 Reveal answer — Q2</summary>

**(a) Correctly-behaving model:** The model should explicitly flag the contradiction — state that one chunk says 14 days and another says 30 days, acknowledge it cannot determine which policy is current, and direct the user to check an authoritative source rather than guessing.

**(b) Without the contradiction instruction:** The model would likely pick the more recent-sounding policy (30 days), combine both into a hedged statement, or return whichever chunk appeared first in the context — producing a confident but potentially wrong answer with no indication of uncertainty.

**(c) Production detection:** Build a contradiction golden set: pairs of inputs with known contradictory chunks and ground-truth expected phrases (e.g., "contradictory", "conflict", "two different policies"). Assert that the model output matches at least one expected phrase. If the pass rate on this golden set drops, the model has regressed to merging contradictions.

</details>

## Judgment question 3

> The system prompt is allocated approximately 54 tokens (use `estimate_tokens(SYSTEM_PROMPT)` to verify). `max_context_tokens` is 3000. The context window for Claude Haiku is 200k tokens.

> You are asked to increase the system prompt to 4000 tokens to add domain-specific instructions and examples. What does this change in the prompt budget, and what is the failure mode if `estimate_tokens()` underestimates by 15%?

In [ ]:
# Your answer:
#
#

<details>
<summary>🔑 Reveal answer — Q3</summary>

**What changes in the prompt budget:** The system prompt grows from ~54 tokens to 4,000 tokens, consuming the entire `max_context_tokens=3000` chunk budget and then some. The chunk-selection logic would see a budget of 3,000 tokens but the system prompt alone already exceeds it, so `truncate_chunks` selects zero chunks — the RAG pipeline degrades to answering from system-prompt context only, with no retrieved evidence.

**Failure mode with 15 % underestimate:** A 4,000-token system prompt is actually ~4,600 tokens by accurate count. If the budget calculation trusts the heuristic, it believes 3,000 tokens are available for chunks but the real available space is 3,000 − 4,600 = −1,600 tokens — i.e., the system prompt already overruns the reserved context, causing the model to truncate or ignore part of the system prompt itself, silently losing instructions.

</details>

## Judgment question 4

> The `estimate_tokens()` function uses a 4-chars-per-token heuristic. The cell above shows both the heuristic estimate and the API's actual token count.

> Under what conditions does the 4-char heuristic *over*-estimate tokens? Under what conditions does it *under*-estimate? Give a concrete example of an input where the heuristic would fail badly.

In [ ]:
# Your answer:
#
# Over-estimates when:
#
# Under-estimates when:
#
# Concrete example:
#

<details>
<summary>🔑 Reveal answer — Q4</summary>

**Over-estimates tokens when:** The text contains many short, high-frequency English words (e.g., "I am in the UK") where a single word is one token but is only 1–3 characters, not 4. The heuristic counts 4+ chars = 1 token, so for compact text it may predict more tokens than actually exist.

**Under-estimates tokens when:** The text contains non-Latin scripts (Chinese, Japanese, Korean, Arabic) where each *character* is typically one token (≈ 1–3 bytes per character), not 4 characters per token. Code-heavy text with many single-character symbols, numbers, and operators also under-estimates because `[`, `]`, `=`, digits each consume a full token.

**Concrete example that fails badly:** A 400-character Chinese sentence. The heuristic estimates 100 tokens (400 / 4). The actual token count is ~400 (one token per character) — a 4× underestimate that would allow the system to accept context that completely overflows the budget.

</details>